In [2]:
import cv2
from ultralytics import YOLO
import numpy as np
model = YOLO("yolo11n.pt")

In [3]:
def class_wise_nms(boxes, classes, confidences, iou_threshold = 0.5):
    #get the unique classes
    unique_classes = np.unique(classes)
    print(unique_classes)
    boxes = np.array(boxes)
    classes = np.array(classes)
    confidences = np.array(confidences)

    nms_boxes, nms_classes, nms_confidences = [], [], []
    
    for each_class in unique_classes:
        #get the boxes of the each_class
        class_boxes = boxes[classes==each_class]
        class_conf = confidences[classes==each_class]
        # print(class_conf)

        while len(class_boxes) > 0:
            #get the box with max confidence
            max_index = np.argmax(class_conf)#position of box with max confidence

            #append the max conf box into the final list
            nms_boxes.append(class_boxes[max_index])
            nms_classes.append(each_class)
            nms_confidences.append(class_conf[max_index])
            ##################################

            current_box = class_boxes[max_index]
            # print(class_boxes)
            iou = compute_iou(current_box, class_boxes)
            
            #get the postion of the boxes with help of filter -- overlapping boxes
            overlapping_idx = np.where(iou > iou_threshold)[0]

            #delete the overlapping boxesand confidences
            #axis = 0 -- delete the rows -- idx
            class_boxes = np.delete(class_boxes, overlapping_idx, axis = 0)
            class_conf = np.delete(class_conf, overlapping_idx)
    return  nms_boxes, nms_classes, nms_confidences

def compute_iou(current_box, boxes):
    #current_box == x1,y1,x2,y2
    #compute the IOU
    intersect_x1 = np.maximum(current_box[0], boxes[:, 0])
    intersect_y1 = np.maximum(current_box[1], boxes[:, 1])
    intersect_x2 = np.minimum(current_box[2], boxes[:, 2])
    intersect_y2 = np.minimum(current_box[3], boxes[:, 3])
    int_width = intersect_x2 - intersect_x1
    int_height = intersect_y2 - intersect_y1
    intersect_area = (int_width * int_height)

    width_cb = current_box[2] - current_box[0]
    height_cb = current_box[3] - current_box[1]
    current_box_area = (width_cb * height_cb)

    boxes_area = (boxes[:,2] - boxes[:,0]) * (boxes[:,3] - boxes[:,1])

    union_area = (current_box_area + boxes_area) - intersect_area
    iou = intersect_area/ union_area
    # print("IOU Matrix :", iou)
    return iou

In [4]:
def process_results(result, img_raw, conf_threshold):
    boxes = result.boxes.xyxy
    classes = result.boxes.cls
    confidences = result.boxes.conf
    class_dict = result.names
    final_boxes, final_classes, final_confidences = [], [], []
    for box, each_class, conf in zip(boxes,classes, confidences):
        x1, y1, x2, y2 = box.cpu().data.numpy().astype("int")
        #print(x1, y1, x2, y2)
        conf = conf.item()

        if conf > conf_threshold:
            each_class = int(each_class.item())
            class_name = class_dict[each_class]
            final_boxes.append([x1, y1, x2, y2])
            final_classes.append(each_class)
            final_confidences.append(conf)
    selected_boxes, selected_classes, selected_confidences = class_wise_nms(final_boxes, 
                                                                             final_classes, 
                                                                             final_confidences, 
                                                                             iou_threshold=0.5)
    for box, each_class, conf in zip(selected_boxes, selected_classes, selected_confidences):
        x1, y1, x2, y2 = box
        #print(x1, y1, x2, y2)
        each_class = int(each_class.item())
        class_name = class_dict[each_class]
        cv2.rectangle(img_raw, (x1, y1), (x2, y2), (0,0,255), 2)
        cv2.putText(img_raw, class_name, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255,0,0), 3)
    return img_raw

In [5]:
def detect(frame):
    results = model.predict(source=frame, conf = 0, iou = 1)
    frame = process_results(results[0], frame, conf_threshold = 0.5)
    return frame

In [7]:


cap = cv2.VideoCapture(0)
is_frame = True
while is_frame:
    is_frame, frame = cap.read()
    if is_frame:
        annotated_frame = detect(frame)
        cv2.namedWindow("Video", cv2.WINDOW_GUI_EXPANDED)
        cv2.imshow("Video", annotated_frame)
        if cv2.waitKey(1) == ord("q"):
            break
    else:
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 77 persons, 3 birds, 19 backpacks, 8 handbags, 26 ties, 4 sports balls, 25 bottles, 1 cup, 34 chairs, 6 couchs, 13 beds, 3 toilets, 9 sinks, 35 refrigerators, 2 books, 18 clocks, 17 toothbrushs, 197.4ms
Speed: 1.8ms preprocess, 197.4ms inference, 194.9ms postprocess per image at shape (1, 3, 480, 640)
[0]


QFontDatabase: Cannot find font directory /home/abhishek/Computer Vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/abhishek/Computer Vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/abhishek/Computer Vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/abhishek/Computer Vision/venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Canno

0: 480x640 109 persons, 6 birds, 19 backpacks, 2 umbrellas, 4 handbags, 16 ties, 17 bottles, 2 cups, 29 chairs, 2 couchs, 15 beds, 2 toilets, 8 sinks, 32 refrigerators, 5 books, 15 clocks, 17 toothbrushs, 280.9ms
Speed: 6.5ms preprocess, 280.9ms inference, 250.7ms postprocess per image at shape (1, 3, 480, 640)
[0]

0: 480x640 72 persons, 4 birds, 16 backpacks, 6 handbags, 21 ties, 1 sports ball, 18 bottles, 3 cups, 32 chairs, 1 couch, 8 beds, 1 dining table, 2 toilets, 1 tv, 1 remote, 12 sinks, 46 refrigerators, 4 books, 21 clocks, 30 toothbrushs, 285.8ms
Speed: 12.4ms preprocess, 285.8ms inference, 315.0ms postprocess per image at shape (1, 3, 480, 640)
[0]

0: 480x640 91 persons, 5 birds, 18 backpacks, 11 handbags, 22 ties, 21 bottles, 1 cup, 40 chairs, 12 beds, 2 dining tables, 4 sinks, 30 refrigerators, 2 books, 22 clocks, 19 toothbrushs, 212.1ms
Speed: 26.8ms preprocess, 212.1ms inference, 361.9ms postprocess per image at shape (1, 3, 480, 640)
[0]

0: 480x640 80 persons, 3 birds